# Cuaderno de pruebas
## Capítulo 1 - Stochastic Calculus for Finance I: The Binomial Asset Pricing Model - Steven E. Shreve

**Autor:** Daniel González Espinosa   
**Año:** 2026  
**Proyecto Árbol Binomial** 

---

### 1. Configuración del Entorno

Preparamos el entorno de trabajo importando las librerías matemáticas estándar y las clases fundamentales de nuestra librería local. 

El motor de valoración que pondremos a prueba (`ReducedStateEngine`) implementa una compresión de estados mediante recombinación de nodos. Esto, combinado con un diseño de polimorfismo puro para las opciones (`EuropeanCall`, `LookbackOption`), nos permitirá validar los principios de no-arbitraje y sensibilidad al precio de Shreve esquivando la explosión combinatoria algorítmica.

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

from binomial_pricer.equity_model import BinomialStockModel
from binomial_pricer.payoffs import EuropeanCall, LookbackOption
from binomial_pricer.engines import ReducedStateEngine

print("✅ Entorno configurado correctamente.")
print("✅ Clases de valoración binomial importadas con éxito.")

### 2. Condición de No-Arbitraje (Ref: Shreve Ejercicio 1.1)

El primer pilar del modelo binomial es asegurar que la configuración matemática del mercado no permita "ganancias sin riesgo", es decir, que excluya el arbitraje. Para un modelo con un factor de subida $u$, un factor de bajada $d$ y una tasa de interés libre de riesgo $r$, el teorema fundamental de valoración requiere estrictamente que 

$$0 < d < 1+r < u.$$

Si violamos estas fronteras analíticas, la estructura del mercado colapsa financieramente:
*   **Si $d \geq 1+r$:** Existe arbitraje pidiendo dinero prestado al banco al tipo $r$ para invertirlo íntegramente en la acción, garantizando una ganancia neta.
*   **Si $u \leq 1+r$:** Existe arbitraje vendiendo en corto la acción para depositar el capital en el banco al tipo $r$.

A continuación, visualizamos computacionalmente el espacio paramétrico válido para una tasa $r$ del 5%.

In [ ]:
r = 0.05
umbral_libre_riesgo = 1 + r

d_values = np.linspace(0.1, 1.5, 500)
u_max = 2.0

fig, ax = plt.subplots()

ax.axvline(x=umbral_libre_riesgo, color='darkred', linestyle='--', 
           label=r'Frontera inferior banco: $d = 1+r$')
ax.axhline(y=umbral_libre_riesgo, color='darkblue', linestyle='--', 
           label=r'Frontera superior banco: $u = 1+r$')
ax.plot([0.1, u_max], [0.1, u_max], color='gray', linestyle=':', 
        label=r'Límite de coherencia del árbol: $u = d$')

ax.fill_between(d_values, umbral_libre_riesgo, u_max, 
                where=(d_values < umbral_libre_riesgo), 
                color='forestgreen', alpha=0.3, 
                label='Región Válida (Sin Arbitraje)')

ax.fill_between(d_values, d_values, u_max, 
                where=(d_values >= umbral_libre_riesgo), 
                color='crimson', alpha=0.15, 
                label='Arbitraje: Préstamo sin riesgo')

ax.fill_between(d_values, d_values, umbral_libre_riesgo, 
                where=(d_values < umbral_libre_riesgo), 
                color='darkorange', alpha=0.2, 
                label='Arbitraje: Venta en corto sin riesgo')

ax.set_xlim(0.5, 1.5)
ax.set_ylim(0.5, 2.0)
ax.set_xlabel('Factor de bajada ($d$)', fontweight='bold')
ax.set_ylabel('Factor de subida ($u$)', fontweight='bold')
ax.set_title(f'Espacio Paramétrico del Modelo Binomial de Shreve (r = {r*100}%)', 
             fontweight='bold', pad=15)
ax.legend(loc='upper left', frameon=True, shadow=True)

plt.tight_layout()
plt.show()


print("Prueba de estrés de la arquitectura (Fronteras de Arbitraje):")

try:
    modelo_invalido = BinomialStockModel(S0=100, u=1.20, d=1.10, r=0.05)
except ValueError as e:
    print(f"✅ Bloqueo de arbitraje exitoso: {e}")

### 3. La Medida de Probabilidad Neutral al Riesgo ($\tilde{p}$)

El núcleo del teorema de valoración de Shreve radica en la construcción de una medida de probabilidad neutral al riesgo ($\tilde{p}$). 

La ecuación fundamental para la probabilidad de una trayectoria alcista en el modelo binomial es

$$ \tilde{p} = \frac{1+r-d}{u-d} .$$

Es crucial observar que la probabilidad histórica o real de que el mercado suba ($p$) no aparece en esta fórmula. El precio de un derivado no depende de nuestras expectativas sobre la dirección del mercado, sino únicamente de la estructura de no-arbitraje dictada por $u$, $d$ y $r$.

A continuación, fijamos la volatilidad del activo subyacente (mediante $u$ y $d$) y evaluamos cómo evoluciona $\tilde{p}$ frente a variaciones en la tasa de interés $r$. Para mantener la coherencia matemática, $r$ variará estrictamente dentro de los límites de no-arbitraje ($d < 1+r < u$).

In [ ]:
u = 1.20  
d = 0.80  

r_min = d - 1
r_max = u - 1

epsilon = 0.001
r_values = np.linspace(r_min + epsilon, r_max - epsilon, 200)


p_tilde_values = []
for r_val in r_values:
    modelo_test = BinomialStockModel(S0=100.0, u=u, d=d, r=r_val)
    
    p_tilde, q_tilde = modelo_test.risk_neutral_prob
    p_tilde_values.append(p_tilde)

fig, ax = plt.subplots()

ax.plot(r_values, p_tilde_values, color='teal', linewidth=2.5, 
        label=r'Evolución de $\tilde{p}$ según $r$')

ax.axhline(y=0, color='crimson', linestyle='--', alpha=0.5, 
           label=r'Frontera inferior ($\tilde{p} \to 0$)')
ax.axhline(y=1, color='crimson', linestyle='--', alpha=0.5, 
           label=r'Frontera superior ($\tilde{p} \to 1$)')

ax.fill_between(r_values, 0, 1, color='teal', alpha=0.05)

ax.set_title(r'Sensibilidad de la Probabilidad Neutral al Riesgo ($\tilde{p}$) frente a $r$', 
             fontweight='bold', pad=15)
ax.set_xlabel('Tasa de interés libre de riesgo ($r$)', fontweight='bold')
ax.set_ylabel(r'Probabilidad Neutral al Riesgo ($\tilde{p}$)', fontweight='bold')

ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0%}'))

ax.legend(loc='upper left', frameon=True, shadow=True)

plt.tight_layout()
plt.show()

### 4. Sensibilidad del Precio y Cobertura Delta ($\Delta$)

Una vez demostrada la coherencia de las fronteras teóricas (no-arbitraje y medida neutral al riesgo), procedemos a instanciar la arquitectura central de nuestra librería `binomial_pricer`. 

Según el teorema fundamental de Shreve, el precio de un derivado $V_0$ se determina calculando el valor esperado descontado bajo la medida $\tilde{p}$. Simultáneamente, el modelo permite calcular la posición exacta en el activo subyacente $\Delta_0$ que forma la cartera de réplica y neutraliza el riesgo de mercado. La sensibilidad Delta en el nodo inicial se define mediante la derivada discreta

$$ \Delta_0 = \frac{V_1(H) - V_1(T)}{S_1(H) - S_1(T)}.$$

Acoplamos `BinomialStockModel` con `ReducedStateEngine` para valorar una familia de opciones `EuropeanCall` bajo un espectro de precios de ejercicio (Strikes, $K$). Analizaremos la transición de la opción desde un estado de ejercicio altamente favorable hasta un estado de ejercicio desfavorable, evaluando cómo decae el precio y cómo se ajusta su ratio de cobertura inicial.

In [ ]:
S0 = 100.0   
u = 1.20     
d = 0.80     
r = 0.05    
N = 10       

modelo_accion = BinomialStockModel(S0, u, d, r)

motor_valoracion = ReducedStateEngine()

strikes = np.linspace(60, 150, 50)
precios_V0 = []
deltas_0 = []

for K in strikes:

    call_europea = EuropeanCall(strike=K)

    resultado = motor_valoracion.price(model=modelo_accion, payoff=call_europea, n_periods=N)
    
    precios_V0.append(resultado.v0)
    deltas_0.append(resultado.delta0)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

ax1.plot(strikes, precios_V0, color='navy', linewidth=2.5)
ax1.axvline(x=S0, color='gray', linestyle=':', label='$S_0$')
ax1.set_title(r'Precio de la Call Europea ($V_0$)', fontweight='bold', pad=10)
ax1.set_xlabel('Precio de Ejercicio ($K$)', fontweight='bold')
ax1.set_ylabel('Valor de la Opción', fontweight='bold')
ax1.fill_between(strikes, precios_V0, color='navy', alpha=0.1)
ax1.legend()

ax2.plot(strikes, deltas_0, color='crimson', linewidth=2.5)
ax2.axvline(x=S0, color='gray', linestyle=':', label='$S_0$')
ax2.set_title(r'Cobertura Delta ($\Delta_0$)', fontweight='bold', pad=10)
ax2.set_xlabel('Precio de Ejercicio ($K$)', fontweight='bold')
ax2.set_ylabel('Coeficiente de Cobertura', fontweight='bold')
ax2.fill_between(strikes, deltas_0, color='crimson', alpha=0.1)
ax2.legend()

plt.tight_layout()
plt.show()

### 5. El Reto de la Trayectoria: Opciones Lookback vs. Asiáticas

Al introducir opciones dependientes de la trayectoria (*path-dependent*), el principio de valoración neutral al riesgo exige ampliar nuestro espacio de estados. Ya no basta con conocer el precio actual del subyacente ($S_n$); necesitamos una variable de estado adicional, $m$, que capture la información histórica relevante de la trayectoria. 

Este proyecto define esta transición mediante la interfaz abstracta `PathDependentPayoff`, delegando a la clase derivada la definición de los métodos `initial_aggregate()`, `update_aggregate()` y `terminal_value()`. Sin embargo, la naturaleza de $m$ dicta si el árbol binomial es computacionalmente tratable:

1. **Opción Lookback:**
   Para una opción Lookback, la variable de estado es el máximo histórico: $M_n = \max_{0 \le i \le n} S_i$. En un árbol donde los factores de subida y bajada son constantes ($u, d$), el precio $S_n$ toma valores discretos en una cuadrícula (lattice). En consecuencia, múltiples trayectorias que convergen al mismo precio $S_n$ también comparten el mismo máximo $M_n$. Esta convergencia permite que nuestro `ReducedStateEngine` colapse ramas redundantes, manteniendo la complejidad algorítmica manejable.

2. **Opción Asiática Aritmética (La suma no recombina):**
   Para una opción Asiática estándar, la variable de estado es la suma acumulada: $Y_n = \sum_{i=0}^n S_i$. A diferencia del máximo, el orden exacto en que ocurren las subidas y bajadas altera el valor de los precios intermedios y, por tanto, altera la suma total. Por ejemplo, una trayectoria (Sube, Baja) frente a (Baja, Sube) termina en el mismo $S_2$, pero genera un $Y_2$ distinto. Dado que casi ninguna trayectoria comparte el mismo $Y_n$, el árbol no recombina y el número de estados crece a un ritmo insostenible de $O(2^N)$.

A continuación, validamos la tratabilidad de la opción Lookback. Tomamos $N=15$ y analizamos las claves generadas (`StateKey`) para demostrar la reducción de estados en comparación con la enumeración de fuerza bruta.

In [ ]:
S0 = 100.0
u = 1.20
d = 0.80
r = 0.05
N = 15  

modelo_accion = BinomialStockModel(S0, u, d, r)
opcion_lookback = LookbackOption()
motor_reducido = ReducedStateEngine()

resultado_lookback = motor_reducido.price(model=modelo_accion, payoff=opcion_lookback, n_periods=N)

estados_n15 = [key for key in resultado_lookback.value_grid.keys() if key[0] == N]
num_estados_reales = len(estados_n15)
rutas_totales = 2 ** N

print("=== ANÁLISIS DE COMPLEJIDAD (N = 15) ===")
print(f"Trayectorias posibles (Enumeración $O(2^N)$): {rutas_totales:,}")
print(f"Estados únicos computados en n=15:            {num_estados_reales:,}")
print(f"Factor de compresión del algoritmo:           {rutas_totales / num_estados_reales:.2f}x\n")

print("=== DEMOSTRACIÓN DE CONVERGENCIA DE ESTADOS ===")
s_intermedio = S0 * (u**8) * (d**7)

estados_mismo_spot = [
    key for key in estados_n15 
    if np.isclose(key[1], s_intermedio)
]

estados_mismo_spot.sort(key=lambda x: x[2])

print(f"Llegando al precio final S_15 = {s_intermedio:.2f}, el árbol solo registra los siguientes estados:")
for n, s, m in estados_mismo_spot:
    print(f" -> Clave: (n={n}, S_n={s:.2f}, Máximo={m:.2f})")

print(f"\nConclusión: De las miles de trayectorias que terminan en S_15 = {s_intermedio:.2f}, ")
print(f"la historia de los máximos ('m') converge en únicamente {len(estados_mismo_spot)} variaciones distintas.")

### 6. Análisis de Complejidad: $O(2^N)$ vs $O(N^2)$

La valoración de derivados mediante árboles binomiales se enfrenta a un muro computacional insalvable si se implementa de manera ingenua. La clase `RecombiningLattice`, mediante su método `enumerate_paths()`, representa el enfoque puramente combinatorio del proceso iterativo. Para un número de periodos $N$, el motor debe evaluar $2^N$ secuencias posibles. Esto hace que el modelo sea matemáticamente correcto, pero algorítmicamente intratable para cualquier $N$ lo suficientemente grande.

Por el contrario, la arquitectura orientada a objetos de `ReducedStateEngine` implementa la compresión del espacio de estados descrita por Shreve. Al reconocer que en un árbol estándar el número de nodos de precios distintos en un instante $n$ es $n+1$, la complejidad espacial y temporal para opciones independientes de la trayectoria (como la Call Europea) se reduce a un orden $O(N^2)$.

A continuación, realizaremos un análisis de rendimiento computacional sometiendo a nuestro motor a un test de estrés con valores de $N$ crecientes (hasta 1000 periodos). Esto demostrará cómo la abstracción de estados transforma un problema exponencial irresoluble en un cálculo que finaliza en fracciones de segundo.

In [ ]:
S0 = 100.0
u = 1.20
d = 0.80
r = 0.05

modelo_accion = BinomialStockModel(S0, u, d, r)
call_europea = EuropeanCall(strike=100.0)
motor_reducido = ReducedStateEngine()

N_values = [10, 50, 100, 250, 500, 1000]
tiempos_ejecucion = []

print("Evaluando ReducedStateEngine (Call Europea)...\n")

for N in N_values:
    inicio = time.perf_counter()
    _ = motor_reducido.price(model=modelo_accion, payoff=call_europea, n_periods=N)
    
    fin = time.perf_counter()
    tiempo_transcurrido = fin - inicio
    tiempos_ejecucion.append(tiempo_transcurrido)
    
    print(f" -> Profundidad N = {N:4d} | Tiempo: {tiempo_transcurrido:.6f} segundos")

fig, ax = plt.subplots(figsize=(12, 6))

ax.plot(N_values, tiempos_ejecucion, marker='o', color='rebeccapurple', 
        linewidth=2.5, markersize=8, label='ReducedStateEngine')

factor_escala = tiempos_ejecucion[-1] / (N_values[-1] ** 2)
tiempos_teoricos_n2 = [factor_escala * (n ** 2) for n in N_values]

ax.plot(N_values, tiempos_teoricos_n2, color='gray', linestyle='--', 
        linewidth=2, label=r'Crecimiento Teórico $O(N^2)$')

ax.set_title('Rendimiento Computacional: Valoración de Call Europea ($O(N^2)$)', 
             fontweight='bold', pad=15)
ax.set_xlabel('Profundidad del Árbol Binomial ($N$)', fontweight='bold')
ax.set_ylabel('Tiempo de Ejecución (segundos)', fontweight='bold')

ax.grid(True, linestyle=':', alpha=0.7)
ax.legend(loc='upper left', frameon=True, shadow=True)

plt.tight_layout()
plt.show()